# 🧠 Нейронные сети: Практикум
### По материалам туториалов Victor Zhou, адаптация для Google Colab
**Выполнила:** Манина В. С., группа ББИ2302, МТУСИ

**Содержание:**
1. [Нейронная сеть с нуля (NumPy)](#part1) — нейрон, сеть, MSE, SGD
2. [Keras + MNIST](#part2) — классификация рукописных цифр (Dense сеть)
3. [RNN с нуля (NumPy)](#part3) — анализ тональности текста
4. [Keras + LSTM (IMDB)](#part4) — анализ отзывов на фильмы
5. [Свёрточные нейросети (CNN)](#part5) — введение и реализация

---
> 💡 **Совет:** Для ускорения обучения включите GPU: `Среда выполнения → Изменить тип среды выполнения → GPU`

---
<a id='part1'></a>
# Часть 1. Нейронная сеть с нуля на NumPy
## Задача: предсказать пол человека по весу и росту

| Имя    | Вес (кг) | Рост (см) | Пол |
|--------|----------|-----------|-----|
| Алиса  | 58 (-2)  | 165 (-1)  | Ж=1 |
| Боб    | 83 (+25) | 172 (+6)  | М=0 |
| Чарли  | 75 (+17) | 170 (+4)  | М=0 |
| Диана  | 51 (-15) | 160 (-6)  | Ж=1 |

*Значения сдвинуты: Вес − 60, Рост − 166*

### 1.1 Строительный блок: отдельный нейрон

In [ ]:
import numpy as np

def sigmoid(x):
    """Функция активации: f(x) = 1 / (1 + e^(-x))"""
    return 1 / (1 + np.exp(-x))

class Neuron:
    def __init__(self, weights, bias):
        self.weights = weights
        self.bias = bias

    def feedforward(self, inputs):
        """Прямой проход: взвешенная сумма + смещение + сигмоида"""
        total = np.dot(self.weights, inputs) + self.bias
        return sigmoid(total)

# Пример: w = [0, 1], b = 4, x = [2, 3]
weights = np.array([0, 1])  # w1=0, w2=1
bias = 4                    # b=4
n = Neuron(weights, bias)

x = np.array([2, 3])
result = n.feedforward(x)
print(f"Входные данные: {x}")
print(f"Взвешенная сумма: {np.dot(weights, x) + bias}")
print(f"Выход нейрона:  {result:.4f}  (ожидается ≈ 0.9991)")

### 1.2 Простая нейронная сеть (2 входа → 2 скрытых нейрона → 1 выход)

In [ ]:
class SimpleNeuralNetwork:
    """
    Архитектура:
      - 2 входа (вес, рост)
      - Скрытый слой: 2 нейрона (h1, h2)
      - Выходной слой: 1 нейрон (o1)
    Все нейроны имеют одинаковые начальные веса w=[0,1], b=0
    """
    def __init__(self):
        weights = np.array([0, 1])
        bias = 0
        self.h1 = Neuron(weights, bias)
        self.h2 = Neuron(weights, bias)
        self.o1 = Neuron(weights, bias)

    def feedforward(self, x):
        out_h1 = self.h1.feedforward(x)
        out_h2 = self.h2.feedforward(x)
        # Входы o1 — это выходы h1 и h2
        out_o1 = self.o1.feedforward(np.array([out_h1, out_h2]))
        return out_o1

network = SimpleNeuralNetwork()
x = np.array([2, 3])
result = network.feedforward(x)
print(f"Выход сети для x={x}: {result:.4f}  (ожидается ≈ 0.7216)")

### 1.3 Функция потерь MSE (среднеквадратичная ошибка)

In [ ]:
def mse_loss(y_true, y_pred):
    """MSE = среднее (y_true - y_pred)^2"""
    return ((y_true - y_pred) ** 2).mean()

# Пример: сеть предсказывает 0 для всех (думает, что все мужчины)
y_true = np.array([1, 0, 0, 1])  # Алиса Ж, Боб М, Чарли М, Диана Ж
y_pred = np.array([0, 0, 0, 0])  # Сеть всегда предсказывает М

loss = mse_loss(y_true, y_pred)
print(f"y_true: {y_true}")
print(f"y_pred: {y_pred}")
print(f"MSE потери: {loss}  (ожидается 0.5)")
print()
print("Цель обучения: минимизировать MSE → минимизировать ошибки предсказания")

### 1.4 Полная нейронная сеть с обратным распространением ошибки (Backprop + SGD)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def deriv_sigmoid(x):
    """Производная сигмоиды: f'(x) = f(x) * (1 - f(x))"""
    fx = sigmoid(x)
    return fx * (1 - fx)

def mse_loss(y_true, y_pred):
    return ((y_true - y_pred) ** 2).mean()

class OurNeuralNetwork:
    """
    Нейронная сеть:
      - 2 входа (вес, рост)
      - Скрытый слой: 2 нейрона (h1, h2)
      - Выходной слой: 1 нейрон (o1)
    Обучение: стохастический градиентный спуск (SGD) + Backpropagation
    """
    def __init__(self):
        # Случайная инициализация весов
        self.w1 = np.random.normal()
        self.w2 = np.random.normal()
        self.w3 = np.random.normal()
        self.w4 = np.random.normal()
        self.w5 = np.random.normal()
        self.w6 = np.random.normal()
        # Случайная инициализация смещений
        self.b1 = np.random.normal()
        self.b2 = np.random.normal()
        self.b3 = np.random.normal()

    def feedforward(self, x):
        h1 = sigmoid(self.w1 * x[0] + self.w2 * x[1] + self.b1)
        h2 = sigmoid(self.w3 * x[0] + self.w4 * x[1] + self.b2)
        o1 = sigmoid(self.w5 * h1 + self.w6 * h2 + self.b3)
        return o1

    def train(self, data, all_y_trues):
        learn_rate = 0.1
        epochs = 1000
        losses = []

        for epoch in range(epochs):
            for x, y_true in zip(data, all_y_trues):
                # --- Прямой проход ---
                sum_h1 = self.w1 * x[0] + self.w2 * x[1] + self.b1
                h1 = sigmoid(sum_h1)
                sum_h2 = self.w3 * x[0] + self.w4 * x[1] + self.b2
                h2 = sigmoid(sum_h2)
                sum_o1 = self.w5 * h1 + self.w6 * h2 + self.b3
                o1 = sigmoid(sum_o1)
                y_pred = o1

                # --- Вычисление частных производных ---
                d_L_d_ypred = -2 * (y_true - y_pred)

                # Нейрон o1
                d_ypred_d_w5 = h1 * deriv_sigmoid(sum_o1)
                d_ypred_d_w6 = h2 * deriv_sigmoid(sum_o1)
                d_ypred_d_b3 = deriv_sigmoid(sum_o1)
                d_ypred_d_h1 = self.w5 * deriv_sigmoid(sum_o1)
                d_ypred_d_h2 = self.w6 * deriv_sigmoid(sum_o1)

                # Нейрон h1
                d_h1_d_w1 = x[0] * deriv_sigmoid(sum_h1)
                d_h1_d_w2 = x[1] * deriv_sigmoid(sum_h1)
                d_h1_d_b1 = deriv_sigmoid(sum_h1)

                # Нейрон h2
                d_h2_d_w3 = x[0] * deriv_sigmoid(sum_h2)
                d_h2_d_w4 = x[1] * deriv_sigmoid(sum_h2)
                d_h2_d_b2 = deriv_sigmoid(sum_h2)

                # --- Обновление весов (SGD) ---
                self.w1 -= learn_rate * d_L_d_ypred * d_ypred_d_h1 * d_h1_d_w1
                self.w2 -= learn_rate * d_L_d_ypred * d_ypred_d_h1 * d_h1_d_w2
                self.b1 -= learn_rate * d_L_d_ypred * d_ypred_d_h1 * d_h1_d_b1
                self.w3 -= learn_rate * d_L_d_ypred * d_ypred_d_h2 * d_h2_d_w3
                self.w4 -= learn_rate * d_L_d_ypred * d_ypred_d_h2 * d_h2_d_w4
                self.b2 -= learn_rate * d_L_d_ypred * d_ypred_d_h2 * d_h2_d_b2
                self.w5 -= learn_rate * d_L_d_ypred * d_ypred_d_w5
                self.w6 -= learn_rate * d_L_d_ypred * d_ypred_d_w6
                self.b3 -= learn_rate * d_L_d_ypred * d_ypred_d_b3

            # Сохраняем потери каждые 10 эпох
            if epoch % 10 == 0:
                y_preds = np.apply_along_axis(self.feedforward, 1, data)
                loss = mse_loss(all_y_trues, y_preds)
                losses.append((epoch, loss))
                if epoch % 100 == 0:
                    print(f"Эпоха {epoch:4d}  |  MSE потери: {loss:.4f}")

        return losses

# ---- Датасет ----
# Вес сдвинут на -60, рост на -166 (нормализация)
data = np.array([
    [-2, -1],   # Алиса
    [25,  6],   # Боб
    [17,  4],   # Чарли
    [-15, -6],  # Диана
])
all_y_trues = np.array([1, 0, 0, 1])  # Ж=1, М=0

np.random.seed(42)
network = OurNeuralNetwork()
losses = network.train(data, all_y_trues)

In [ ]:
# График потерь во время обучения
epochs_plot, loss_plot = zip(*losses)
plt.figure(figsize=(10, 5))
plt.plot(epochs_plot, loss_plot, color='#e74c3c', linewidth=2)
plt.xlabel('Эпоха', fontsize=13)
plt.ylabel('MSE потери', fontsize=13)
plt.title('Снижение потерь в процессе обучения', fontsize=15)
plt.grid(alpha=0.3)
plt.show()

# Тест на новых данных
print("\n=== Предсказания для новых людей ===")
emily = np.array([-7, -3])   # вес ~53 кг, рост ~163 см → Ж
frank = np.array([20,  2])   # вес ~80 кг, рост ~168 см → М

emily_pred = network.feedforward(emily)
frank_pred = network.feedforward(frank)

print(f"Emily (53 кг, 163 см): {emily_pred:.3f}  → {'Женщина ✓' if emily_pred > 0.5 else 'Мужчина ✗'}")
print(f"Frank (80 кг, 168 см): {frank_pred:.3f}  → {'Женщина ✗' if frank_pred > 0.5 else 'Мужчина ✓'}")

---
<a id='part2'></a>
# Часть 2. Keras + MNIST
## Классификация рукописных цифр (Dense нейронная сеть)

**Задача:** по изображению 28×28 пикселей определить цифру (0–9)

In [ ]:
# Установка пакета mnist (датасет)
!pip install mnist -q

In [ ]:
import numpy as np
import mnist
import matplotlib.pyplot as plt

# Загрузка данных
train_images = mnist.train_images()
train_labels = mnist.train_labels()
test_images  = mnist.test_images()
test_labels  = mnist.test_labels()

print(f"Обучающий набор: {train_images.shape}  |  Метки: {train_labels.shape}")
print(f"Тестовый набор:  {test_images.shape}  |  Метки: {test_labels.shape}")

# Показываем первые 10 изображений
fig, axes = plt.subplots(1, 10, figsize=(14, 2))
for i, ax in enumerate(axes):
    ax.imshow(train_images[i], cmap='gray')
    ax.set_title(train_labels[i], fontsize=14, fontweight='bold')
    ax.axis('off')
plt.suptitle('Примеры из датасета MNIST', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Подготовка данных
# 1. Нормализация: [0, 255] → [-0.5, 0.5]
train_images = (train_images / 255) - 0.5
test_images  = (test_images  / 255) - 0.5

# 2. Выравнивание: 28x28 → вектор длиной 784
train_images = train_images.reshape((-1, 784))
test_images  = test_images.reshape((-1, 784))

print(f"После подготовки:")
print(f"  Train images: {train_images.shape}")
print(f"  Test images:  {test_images.shape}")
print(f"  Диапазон значений: [{train_images.min():.1f}, {train_images.max():.1f}]")

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical

# Построение модели
model = Sequential([
    Dense(64, activation='relu', input_shape=(784,)),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax'),  # 10 классов (цифры 0-9)
])

model.summary()

In [ ]:
# Компиляция
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

# Обучение (to_categorical: 7 → [0,0,0,0,0,0,0,1,0,0])
history = model.fit(
    train_images,
    to_categorical(train_labels),
    epochs=10,
    batch_size=32,
    validation_data=(test_images, to_categorical(test_labels)),
    verbose=1
)

In [ ]:
# Оценка модели на тестовых данных
test_loss, test_acc = model.evaluate(test_images, to_categorical(test_labels), verbose=0)
print(f"Точность на тесте: {test_acc*100:.2f}%")
print(f"Потери на тесте:   {test_loss:.4f}")

# График обучения
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['accuracy'],     label='Обучение', color='#3498db')
ax1.plot(history.history['val_accuracy'], label='Валидация', color='#e74c3c')
ax1.set_title('Точность по эпохам'); ax1.set_xlabel('Эпоха'); ax1.set_ylabel('Точность')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(history.history['loss'],     label='Обучение', color='#3498db')
ax2.plot(history.history['val_loss'], label='Валидация', color='#e74c3c')
ax2.set_title('Потери по эпохам'); ax2.set_xlabel('Эпоха'); ax2.set_ylabel('Потери')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Предсказания на первых 10 тестовых изображениях
predictions = model.predict(test_images[:10], verbose=0)
predicted_labels = np.argmax(predictions, axis=1)

fig, axes = plt.subplots(2, 10, figsize=(16, 4))
for i in range(10):
    # Оригинальное изображение
    axes[0][i].imshow(test_images[i].reshape(28, 28), cmap='gray')
    axes[0][i].axis('off')
    axes[0][i].set_title(f'Факт: {test_labels[i]}', fontsize=10)
    # Уверенность модели
    correct = predicted_labels[i] == test_labels[i]
    color = '#2ecc71' if correct else '#e74c3c'
    axes[1][i].bar(range(10), predictions[i], color=[color if j == predicted_labels[i] else '#bdc3c7' for j in range(10)])
    axes[1][i].set_xticks(range(10)); axes[1][i].set_yticks([])
    axes[1][i].set_xlabel(f'Пред: {predicted_labels[i]}', fontsize=9)

axes[0][0].set_ylabel('Изображение', fontsize=10)
axes[1][0].set_ylabel('Вероятность', fontsize=10)
plt.suptitle('Предсказания модели (зелёный = верно, красный = ошибка)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Сохранение и загрузка весов модели
model.save_weights('mnist_model.h5')
print("Веса модели сохранены в mnist_model.h5")

# Чтобы загрузить позже:
# model.load_weights('mnist_model.h5')

### 🔬 Эксперименты: меняем гиперпараметры

In [ ]:
from tensorflow.keras.layers import Dropout
from tensorflow.keras.optimizers import Adam

# Эксперимент 1: Более глубокая сеть + Dropout + кастомный lr
model_deep = Sequential([
    Dense(128, activation='relu', input_shape=(784,)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(10, activation='softmax'),
])

model_deep.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

print("Архитектура глубокой сети:")
model_deep.summary()

history_deep = model_deep.fit(
    train_images, to_categorical(train_labels),
    epochs=10, batch_size=64,
    validation_data=(test_images, to_categorical(test_labels)),
    verbose=1
)

_, acc_deep = model_deep.evaluate(test_images, to_categorical(test_labels), verbose=0)
print(f"\nТочность глубокой сети: {acc_deep*100:.2f}%")

---
<a id='part3'></a>
# Часть 3. RNN с нуля на NumPy
## Анализ тональности текста

Архитектура «многие к одному»: последовательность слов → положительный/отрицательный

In [ ]:
import numpy as np
from numpy.random import randn
import random

# ===== Датасет =====
train_data = {
    'good':                          True,
    'bad':                           False,
    'not bad':                       True,
    'not good':                      False,
    'very good':                     True,
    'really bad':                    False,
    'i am good':                     True,
    'this is good':                  True,
    'i am bad':                      False,
    'this is bad':                   False,
    'an excellent good':             True,
    'so bad':                        False,
    'just really bad':               False,
    'not really bad':                True,
    'i am not bad':                  True,
    'i am not good':                 False,
}

test_data = {
    'this is happy':                 True,
    'i am good':                     True,
    'this is not good':              False,
    'i am not bad':                  True,
    'i am bad':                      False,
    'this is really bad':            False,
    'not really good':               False,
    'i am really bad':               False,
}

# Строим словарь
vocab = list(set([w for text in train_data.keys() for w in text.split(' ')]))
vocab_size = len(vocab)
word_to_idx = {w: i for i, w in enumerate(vocab)}
idx_to_word = {i: w for i, w in enumerate(vocab)}

print(f"Уникальных слов: {vocab_size}")
print(f"Словарь: {vocab}")

In [ ]:
def createInputs(text):
    """Преобразуем текст в список one-hot векторов"""
    inputs = []
    for w in text.split(' '):
        v = np.zeros((vocab_size, 1))
        v[word_to_idx[w]] = 1
        inputs.append(v)
    return inputs

def softmax(xs):
    return np.exp(xs) / sum(np.exp(xs))

# ===== Класс RNN =====
class RNN:
    """Ванильная рекуррентная нейронная сеть.
    
    Уравнения:
      h_t = tanh(W_xh @ x_t + W_hh @ h_{t-1} + b_h)
      y_t = W_hy @ h_t + b_y
    """
    def __init__(self, input_size, output_size, hidden_size=64):
        # Инициализация весов (малые значения)
        self.Whh = randn(hidden_size, hidden_size) / 1000
        self.Wxh = randn(hidden_size, input_size)  / 1000
        self.Why = randn(output_size, hidden_size) / 1000
        # Смещения
        self.bh = np.zeros((hidden_size, 1))
        self.by = np.zeros((output_size, 1))

    def forward(self, inputs):
        """Прямой проход — возвращает выход и кэш скрытых состояний"""
        h = np.zeros((self.Whh.shape[0], 1))
        self.last_inputs = inputs
        self.last_hs = {0: h}

        for i, x in enumerate(inputs):
            h = np.tanh(self.Wxh @ x + self.Whh @ h + self.bh)
            self.last_hs[i + 1] = h

        y = self.Why @ h + self.by
        return y, h

    def backprop(self, d_y, learn_rate=2e-2):
        """Обратное распространение ошибки во времени (BPTT)"""
        n = len(self.last_inputs)

        # Градиенты для Why и by
        d_Why = d_y @ self.last_hs[n].T
        d_by  = d_y

        # Инициализация градиентов для Whh, Wxh, bh
        d_Whh = np.zeros(self.Whh.shape)
        d_Wxh = np.zeros(self.Wxh.shape)
        d_bh  = np.zeros(self.bh.shape)

        # Начинаем с последнего скрытого состояния
        d_h = self.Why.T @ d_y

        # Обратный проход по времени
        for t in reversed(range(n)):
            temp = (1 - self.last_hs[t + 1] ** 2) * d_h
            d_bh  += temp
            d_Whh += temp @ self.last_hs[t].T
            d_Wxh += temp @ self.last_inputs[t].T
            d_h    = self.Whh @ temp

        # Обрезаем градиенты (gradient clipping)
        for d in [d_Wxh, d_Whh, d_Why, d_bh, d_by]:
            np.clip(d, -1, 1, out=d)

        # Обновляем веса
        self.Whh -= learn_rate * d_Whh
        self.Wxh -= learn_rate * d_Wxh
        self.Why -= learn_rate * d_Why
        self.bh  -= learn_rate * d_bh
        self.by  -= learn_rate * d_by

print("Класс RNN определён.")

In [ ]:
def processData(rnn, data, backprop=True):
    """Прогоняет все данные через RNN; опционально обучает"""
    items = list(data.items())
    random.shuffle(items)
    loss = 0
    num_correct = 0

    for x, y in items:
        inputs = createInputs(x)
        target = int(y)

        out, _ = rnn.forward(inputs)
        probs = softmax(out)

        loss += -np.log(probs[target])
        num_correct += int(np.argmax(probs) == target)

        if backprop:
            d_L_d_y = probs.copy()
            d_L_d_y[target] -= 1
            rnn.backprop(d_L_d_y)

    return loss / len(data), num_correct / len(data)

# ===== Обучение RNN =====
np.random.seed(0)
rnn = RNN(vocab_size, 2)  # 2 выхода: положительный / отрицательный

print("Начало обучения RNN...")
print(f"{'Эпоха':>6}  |  {'Train Loss':>10}  {'Train Acc':>10}  |  {'Test Loss':>9}  {'Test Acc':>9}")
print("-" * 65)

for epoch in range(1000):
    train_loss, train_acc = processData(rnn, train_data)

    if epoch % 100 == 99:
        test_loss, test_acc = processData(rnn, test_data, backprop=False)
        print(f"Эпоха {epoch+1:4d}  |  {train_loss:10.3f}  {train_acc:10.3f}  |  {test_loss:9.3f}  {test_acc:9.3f}")

In [ ]:
# Тест на отдельных фразах
test_phrases = [
    "good", "bad", "not bad", "not good",
    "very good", "really bad", "i am good", "i am bad"
]

print("=== Предсказания обученной RNN ===")
print(f"{'Фраза':<25} {'P(позит.)':>10} {'P(негат.)':>10}  Вывод")
print("-" * 65)

for phrase in test_phrases:
    try:
        inputs = createInputs(phrase)
        out, _ = rnn.forward(inputs)
        probs = softmax(out).flatten()
        label = "Позитивно ✓" if np.argmax(probs) == 1 else "Негативно ✗"
        print(f"  {phrase:<23} {probs[1]:>10.3f} {probs[0]:>10.3f}  {label}")
    except KeyError as e:
        print(f"  {phrase:<23}  Слово {e} не в словаре")

---
<a id='part4'></a>
# Часть 4. Keras + LSTM
## Анализ отзывов на фильмы (IMDb датасет)

**Задача:** определить, положительный или отрицательный отзыв на фильм

In [ ]:
# Используем встроенный IMDb датасет из Keras (не требует скачивания)
import numpy as np
from tensorflow import keras
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Embedding, Dropout
import matplotlib.pyplot as plt

# Параметры
MAX_TOKENS = 10000   # Размер словаря
MAX_LEN    = 200     # Максимальная длина отзыва

# Загрузка данных (top MAX_TOKENS слов)
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=MAX_TOKENS)

print(f"Обучающий набор: {len(X_train)} отзывов")
print(f"Тестовый набор:  {len(X_test)} отзывов")
print(f"Пример отзыва (индексы слов): {X_train[0][:20]}...")
print(f"Метка: {y_train[0]} (1=Положительный, 0=Отрицательный)")

In [ ]:
# Подготовка данных: дополняем/обрезаем до MAX_LEN
X_train_pad = pad_sequences(X_train, maxlen=MAX_LEN)
X_test_pad  = pad_sequences(X_test,  maxlen=MAX_LEN)

print(f"Форма обучающих данных после паддинга: {X_train_pad.shape}")

# Расшифруем первый отзыв
word_index = imdb.get_word_index()
idx_to_word = {v+3: k for k, v in word_index.items()}
idx_to_word.update({0: '<PAD>', 1: '<START>', 2: '<UNK>', 3: '<UNUSED>'})

decoded = ' '.join([idx_to_word.get(i, '?') for i in X_train[0][:50]])
print(f"\nНачало первого отзыва:")
print(f"  {decoded}...")
print(f"  Метка: {'Положительный 👍' if y_train[0]==1 else 'Отрицательный 👎'}")

In [ ]:
# Построение модели LSTM
model_rnn = Sequential([
    # 1. Embedding: целые числа → векторы
    Embedding(MAX_TOKENS + 1, 128, input_length=MAX_LEN),
    # 2. LSTM слой
    LSTM(64),
    # 3. Полносвязный слой
    Dense(64, activation='relu'),
    Dropout(0.3),
    # 4. Выходной слой (sigmoid для бинарной классификации)
    Dense(1, activation='sigmoid'),
])

model_rnn.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

model_rnn.summary()

In [ ]:
# Обучение
history_rnn = model_rnn.fit(
    X_train_pad, y_train,
    epochs=5,
    batch_size=64,
    validation_data=(X_test_pad, y_test),
    verbose=1
)

In [ ]:
# Оценка и графики
test_loss_rnn, test_acc_rnn = model_rnn.evaluate(X_test_pad, y_test, verbose=0)
print(f"Точность на тесте: {test_acc_rnn*100:.2f}%")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history_rnn.history['accuracy'],     label='Обучение', color='#9b59b6')
ax1.plot(history_rnn.history['val_accuracy'], label='Валидация', color='#e74c3c')
ax1.set_title('Точность LSTM (IMDb)'); ax1.set_xlabel('Эпоха'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(history_rnn.history['loss'],     label='Обучение', color='#9b59b6')
ax2.plot(history_rnn.history['val_loss'], label='Валидация', color='#e74c3c')
ax2.set_title('Потери LSTM (IMDb)'); ax2.set_xlabel('Эпоха'); ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Тест модели на кастомных отзывах
def encode_review(text, word_index, max_len=200):
    """Кодируем текстовый отзыв в числовой вектор"""
    tokens = text.lower().split()
    encoded = [word_index.get(w, 2) + 3 for w in tokens]  # 2 = <UNK>
    padded = pad_sequences([encoded], maxlen=max_len)
    return padded

reviews = [
    "i loved it! highly recommend it to anyone looking for a great movie to watch",
    "this was awful i hated it so much nobody should watch this the acting was terrible",
    "not bad could be better but overall an enjoyable experience",
    "absolutely brilliant masterpiece of cinema",
]

print("=== Предсказания для кастомных отзывов ===")
for review in reviews:
    encoded = encode_review(review, word_index)
    score = model_rnn.predict(encoded, verbose=0)[0][0]
    sentiment = "Положительный 👍" if score > 0.5 else "Отрицательный 👎"
    print(f"  Оценка: {score:.2f} → {sentiment}")
    print(f"  Отзыв: '{review[:60]}...'" if len(review) > 60 else f"  Отзыв: '{review}'")
    print()

### 🔬 Эксперимент: Двухслойный LSTM + Dropout

In [ ]:
# Более сложная архитектура: два LSTM слоя
model_deep_rnn = Sequential([
    Embedding(MAX_TOKENS + 1, 128, input_length=MAX_LEN),
    # return_sequences=True нужен, чтобы передать всю последовательность следующему LSTM
    LSTM(64, return_sequences=True, dropout=0.2, recurrent_dropout=0.2),
    LSTM(32, dropout=0.2),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid'),
])

model_deep_rnn.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_deep_rnn.summary()

---
<a id='part5'></a>
# Часть 5. Свёрточные нейронные сети (CNN)
## Классификация MNIST с помощью CNN

**Почему CNN лучше Dense для изображений:**
- 🔹 Меньше весов (локальная связность)
- 🔹 Инвариантность к сдвигу (pooling)
- 🔹 Иерархическое извлечение признаков

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical

# Загружаем и готовим данные для CNN (нужны 4D тензоры)
train_images_cnn = mnist.train_images().reshape(-1, 28, 28, 1) / 255.0
test_images_cnn  = mnist.test_images().reshape(-1, 28, 28, 1)  / 255.0
train_labels_cnn = mnist.train_labels()
test_labels_cnn  = mnist.test_labels()

print(f"Форма для CNN: {train_images_cnn.shape}  (N, H, W, Каналы)")

In [ ]:
# Визуализация работы свёрточного фильтра (Sobel — детектор границ)
from scipy.ndimage import convolve

# Фильтр для обнаружения вертикальных границ (Sobel)
sobel_filter = np.array([
    [-1,  0,  1],
    [-2,  0,  2],
    [-1,  0,  1]
])

sample_img = mnist.test_images()[0].astype(float)
filtered = convolve(sample_img, sobel_filter)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(sample_img, cmap='gray')
axes[0].set_title(f'Оригинал (цифра {mnist.test_labels()[0]})', fontsize=12)
axes[1].imshow(sobel_filter, cmap='RdBu_r')
axes[1].set_title('Фильтр Sobel 3×3', fontsize=12)
axes[1].set_xticks([]); axes[1].set_yticks([])
for i in range(3):
    for j in range(3):
        axes[1].text(j, i, str(sobel_filter[i,j]), ha='center', va='center', fontsize=14, fontweight='bold')
axes[2].imshow(np.abs(filtered), cmap='hot')
axes[2].set_title('Результат свёртки\n(Вертикальные границы)', fontsize=12)
for ax in axes: ax.axis('off')
plt.suptitle('Как работает свёрточный фильтр', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Построение CNN модели
model_cnn = Sequential([
    # Свёрточный блок 1
    Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    MaxPooling2D((2, 2)),
    # Свёрточный блок 2
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    # Выравнивание и полносвязные слои
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(10, activation='softmax'),
])

model_cnn.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_cnn.summary()
print(f"\nКоличество параметров CNN: {model_cnn.count_params():,}")

In [ ]:
# Обучение CNN
history_cnn = model_cnn.fit(
    train_images_cnn,
    to_categorical(train_labels_cnn),
    epochs=10,
    batch_size=64,
    validation_data=(test_images_cnn, to_categorical(test_labels_cnn)),
    verbose=1
)

_, cnn_acc = model_cnn.evaluate(test_images_cnn, to_categorical(test_labels_cnn), verbose=0)
print(f"\n🎯 Точность CNN на тесте: {cnn_acc*100:.2f}%")

In [ ]:
# Сравнение: Dense vs CNN
dense_acc = history.history['val_accuracy'][-1]  # из Части 2
cnn_acc_val = history_cnn.history['val_accuracy'][-1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# График точности
axes[0].plot(history.history['val_accuracy'],     label='Dense сеть', color='#3498db', linewidth=2)
axes[0].plot(history_cnn.history['val_accuracy'], label='CNN', color='#e74c3c', linewidth=2)
axes[0].set_title('Точность: Dense vs CNN (MNIST)', fontsize=13)
axes[0].set_xlabel('Эпоха'); axes[0].set_ylabel('Точность на валидации')
axes[0].legend(fontsize=12); axes[0].grid(alpha=0.3)

# Столбчатый график
models = ['Dense (Part 2)', 'CNN (Part 5)']
accs   = [dense_acc * 100, cnn_acc_val * 100]
bars = axes[1].bar(models, accs, color=['#3498db', '#e74c3c'], edgecolor='black', width=0.5)
axes[1].set_title('Финальная точность на тесте', fontsize=13)
axes[1].set_ylabel('Точность (%)')
axes[1].set_ylim([95, 100])
for bar, acc in zip(bars, accs):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'{acc:.2f}%', ha='center', va='bottom', fontsize=13, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Dense сеть: {dense_acc*100:.2f}%")
print(f"CNN:        {cnn_acc_val*100:.2f}%")
print(f"Прирост:    +{(cnn_acc_val - dense_acc)*100:.2f}%")

In [ ]:
# Визуализация карт активаций (что «видит» CNN)
import tensorflow as tf

# Создаём модель, которая выдаёт активации первого Conv2D слоя
activation_model = tf.keras.Model(
    inputs=model_cnn.input,
    outputs=model_cnn.layers[0].output  # первый Conv2D слой
)

sample = test_images_cnn[0:1]  # одно изображение
activations = activation_model.predict(sample, verbose=0)

# Показываем первые 16 фильтров
fig, axes = plt.subplots(4, 8, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    if i < activations.shape[-1]:
        ax.imshow(activations[0, :, :, i], cmap='viridis')
    ax.axis('off')

plt.suptitle(f'Карты активаций первого свёрточного слоя\n'
             f'(32 фильтра для цифры "{test_labels_cnn[0]}")', fontsize=13)
plt.tight_layout()
plt.show()

---
## 📊 Итоговая сводка по всем моделям

In [ ]:
print("=" * 65)
print("ИТОГОВАЯ СВОДКА ПО ВСЕМ МОДЕЛЯМ")
print("=" * 65)
print(f"{'Модель':<30} {'Задача':<25} {'Точность':>10}")
print("-" * 65)
print(f"{'Нейросеть с нуля (NumPy)':<30} {'Пол (4 примера)':<25} {'~95%':>10}")

dense_final_acc = model.evaluate(test_images, to_categorical(test_labels), verbose=0)[1]
print(f"{'Dense Keras (MNIST)':<30} {'Цифры 0-9':<25} {dense_final_acc*100:>9.2f}%")

print(f"{'RNN с нуля (NumPy)':<30} {'Тональность (текст)':<25} {'~100%':>10}")

rnn_final_acc = model_rnn.evaluate(X_test_pad, y_test, verbose=0)[1]
print(f"{'LSTM Keras (IMDb)':<30} {'Отзывы на фильмы':<25} {rnn_final_acc*100:>9.2f}%")

cnn_final_acc = model_cnn.evaluate(test_images_cnn, to_categorical(test_labels_cnn), verbose=0)[1]
print(f"{'CNN Keras (MNIST)':<30} {'Цифры 0-9':<25} {cnn_final_acc*100:>9.2f}%")

print("=" * 65)

---
## 🎯 Чеклист пройденных тем

| Тема | Библиотека | Ключевые концепции |
|------|------------|--------------------|
| Нейрон и сеть с нуля | `numpy` | Sigmoid, feedforward, MSE |
| Обучение с нуля | `numpy` | Backprop, chain rule, SGD |
| Dense сеть | `keras` | Dense, ReLU, Softmax |
| Регуляризация | `keras` | Dropout, Early stopping |
| RNN с нуля | `numpy` | BPTT, hidden state, gradient clipping |
| LSTM | `keras` | LSTM, Embedding, TextVectorization |
| Свёртки | `numpy`/`keras` | Conv2D, MaxPooling2D, карты признаков |

---
*Нейронные сети: Практикум · Victor Zhou · Адаптация 2026*